[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/092.%20The%20Location-Routing%20Problem%20%28LRP%29/P92-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 92. The Location-Routing Problem (LRP)
## Tier 2: The Classic Heuristic (GRASP)

### Key assumptions
- GRASP uses randomized greedy construction for solution diversity
- Local search improves solutions through iterative refinement
- Multiple iterations provide different high-quality solutions
- Restricted Candidate List (RCL) balances greediness and randomness
- 2-opt and exchange operations for route improvement

### Approach (step-by-step)
1. **GRASP Framework**: Implement the two-phase metaheuristic approach
2. **Construction Phase**: Build feasible solutions using RCL-based selection
3. **Local Search Phase**: Improve solutions through neighborhood operations
4. **Iteration Management**: Run multiple GRASP iterations and track best solution
5. **Performance Analysis**: Compare with exact methods and analyze convergence
6. **Visualization**: Show solution quality improvement over iterations

### What to look for in the results
- Solution quality gap compared to optimal solution
- Convergence behavior over GRASP iterations
- Impact of RCL parameter α on solution diversity
- Computational efficiency vs exact methods
- Consistency of solutions across different runs

### Concrete example (from the source)
- **Problem**: Same 2-depot, 3-customer instance as Tier 1
- **GRASP Parameters**: α=0.3 for RCL, 100 iterations
- **Local Search**: 2-opt for routes, exchange for customer-depot assignments
- **Expected Performance**: Near-optimal solutions in fraction of MIP time

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
import itertools
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

In [ ]:
@dataclass
class LRPInstance:
    """Data class for Location-Routing Problem instance"""
    customers: List[int]
    depots: List[int]
    vehicles: List[int]
    depot_costs: Dict[int, float]
    demands: Dict[int, float]
    vehicle_capacity: float
    travel_costs: Dict[Tuple[int, int], float]
    
    def get_all_nodes(self):
        return self.customers + self.depots

@dataclass
class LRPSolution:
    """Data class for LRP solution"""
    opened_depots: List[int]
    customer_assignments: Dict[int, int]
    routes: Dict[int, List[int]]
    total_cost: float
    fixed_cost: float
    routing_cost: float

# Create the concrete example
def create_concrete_example():
    customers = [1, 2, 3]
    depots = [4, 5]
    vehicles = [1, 2]
    
    depot_costs = {4: 100, 5: 120}
    demands = {1: 10, 2: 15, 3: 20}
    vehicle_capacity = 30
    
    travel_costs = {
        (1, 2): 15, (2, 1): 15,
        (1, 3): 20, (3, 1): 20,
        (2, 3): 18, (3, 2): 18,
        (4, 1): 12, (1, 4): 12,
        (4, 2): 14, (2, 4): 14,
        (4, 3): 25, (3, 4): 25,
        (5, 1): 18, (1, 5): 18,
        (5, 2): 16, (2, 5): 16,
        (5, 3): 22, (3, 5): 22,
        (4, 5): 30, (5, 4): 30,
        (1, 1): 0, (2, 2): 0, (3, 3): 0,
        (4, 4): 0, (5, 5): 0
    }
    
    return LRPInstance(
        customers=customers,
        depots=depots,
        vehicles=vehicles,
        depot_costs=depot_costs,
        demands=demands,
        vehicle_capacity=vehicle_capacity,
        travel_costs=travel_costs
    )

instance = create_concrete_example()
print(f"LRP Instance created for GRASP:")
print(f"Customers: {instance.customers}")
print(f"Potential Depots: {instance.depots}")
print(f"Vehicles: {instance.vehicles}")
print(f"Vehicle Capacity: {instance.vehicle_capacity}")

LRP Instance created for GRASP:
Customers: [1, 2, 3]
Potential Depots: [4, 5]
Vehicles: [1, 2]
Vehicle Capacity: 30


In [2]:
def calculate_route_cost(route, instance):
    """Calculate the cost of a single route"""
    if len(route) < 2:
        return 0
    
    total_cost = 0
    for i in range(len(route) - 1):
        u, v = route[i], route[i + 1]
        total_cost += instance.travel_costs.get((u, v), float('inf'))
    return total_cost

def calculate_solution_cost(solution, instance):
    """Calculate total cost of a solution"""
    # Fixed depot costs
    fixed_cost = sum(instance.depot_costs[j] for j in solution.opened_depots)
    
    # Routing costs
    routing_cost = 0
    for k, route in solution.routes.items():
        routing_cost += calculate_route_cost(route, instance)
    
    total_cost = fixed_cost + routing_cost
    return total_cost, fixed_cost, routing_cost

def is_route_feasible(route, instance):
    """Check if a route respects vehicle capacity"""
    total_demand = sum(instance.demands[i] for i in route if i in instance.customers)
    return total_demand <= instance.vehicle_capacity

def greedy_route_construction(customers, depot, instance):
    """Construct a single route using nearest neighbor heuristic"""
    if not customers:
        return []
    
    route = [depot]
    unvisited = customers.copy()
    
    while unvisited:
        current = route[-1]
        # Find nearest unvisited customer
        nearest = min(unvisited, 
                    key=lambda c: instance.travel_costs.get((current, c), float('inf')))
        
        # Check capacity constraint
        temp_route = route + [nearest]
        if is_route_feasible(temp_route, instance):
            route.append(nearest)
            unvisited.remove(nearest)
        else:
            # Return to depot if capacity would be exceeded
            break
    
    # Return to depot
    if route[-1] != depot:
        route.append(depot)
    
    return route, unvisited

print("Utility functions defined for GRASP")

Utility functions defined for GRASP


In [3]:
def construct_rcl(costs, alpha):
    """Build Restricted Candidate List (RCL)"""
    if not costs:
        return []
    
    min_cost = min(costs.values())
    max_cost = max(costs.values())
    
    if min_cost == max_cost:
        return list(costs.keys())
    
    cutoff = min_cost + alpha * (max_cost - min_cost)
    rcl = [candidate for candidate, cost in costs.items() if cost <= cutoff]
    
    return rcl

def grasp_construction_phase(instance, alpha=0.3):
    """GRASP construction phase"""
    # Step 1: Select depots to open using RCL
    depot_costs_eval = {}
    for depot in instance.depots:
        # Estimate cost as fixed cost + average routing cost to customers
        avg_routing_cost = np.mean([instance.travel_costs.get((depot, c), float('inf')) 
                                   for c in instance.customers])
        depot_costs_eval[depot] = instance.depot_costs[depot] + len(instance.customers) * avg_routing_cost
    
    depot_rcl = construct_rcl(depot_costs_eval, alpha)
    selected_depot = random.choice(depot_rcl)
    opened_depots = [selected_depot]
    
    # Step 2: Assign customers to depots and build routes
    unassigned_customers = instance.customers.copy()
    customer_assignments = {}
    routes = {}
    vehicle_id = 1
    
    while unassigned_customers:
        # Select depot for next vehicle
        depot_costs_current = {}
        for depot in opened_depots:
            # Find closest unassigned customer to this depot
            if unassigned_customers:
                closest_customer = min(unassigned_customers, 
                                    key=lambda c: instance.travel_costs.get((depot, c), float('inf')))
                depot_costs_current[depot] = instance.travel_costs.get((depot, closest_customer), float('inf'))
        
        if not depot_costs_current:
            break
            
        depot_rcl = construct_rcl(depot_costs_current, alpha)
        selected_depot = random.choice(depot_rcl)
        
        # Find customers that can be served from this depot
        depot_customers = [c for c in unassigned_customers 
                          if instance.travel_costs.get((selected_depot, c), float('inf')) < float('inf')]
        
        if not depot_customers:
            # Open a new depot if available
            remaining_depots = [d for d in instance.depots if d not in opened_depots]
            if remaining_depots:
                new_depot = random.choice(remaining_depots)
                opened_depots.append(new_depot)
                selected_depot = new_depot
                depot_customers = unassigned_customers.copy()
            else:
                break
        
        # Build route from selected depot
        route, remaining_customers = greedy_route_construction(depot_customers, selected_depot, instance)
        
        if route and len(route) > 2:  # At least depot -> customer -> depot
            routes[vehicle_id] = route
            vehicle_id += 1
            
            # Update assignments
            for customer in route:
                if customer in instance.customers:
                    customer_assignments[customer] = selected_depot
                    if customer in unassigned_customers:
                        unassigned_customers.remove(customer)
        else:
            # Try individual customer routes
            for customer in depot_customers[:]:
                if instance.demands[customer] <= instance.vehicle_capacity:
                    route = [selected_depot, customer, selected_depot]
                    routes[vehicle_id] = route
                    vehicle_id += 1
                    customer_assignments[customer] = selected_depot
                    unassigned_customers.remove(customer)
                    break
    
    # Create solution object
    solution = LRPSolution(
        opened_depots=opened_depots,
        customer_assignments=customer_assignments,
        routes=routes,
        total_cost=0,
        fixed_cost=0,
        routing_cost=0
    )
    
    # Calculate costs
    total_cost, fixed_cost, routing_cost = calculate_solution_cost(solution, instance)
    solution.total_cost = total_cost
    solution.fixed_cost = fixed_cost
    solution.routing_cost = routing_cost
    
    return solution

print("GRASP construction phase implemented")

GRASP construction phase implemented


In [4]:
def two_opt_route_improvement(route, instance):
    """Improve a route using 2-opt local search"""
    if len(route) <= 3:  # Need at least 4 nodes for meaningful 2-opt
        return route
    
    best_route = route.copy()
    best_cost = calculate_route_cost(best_route, instance)
    
    improved = True
    while improved:
        improved = False
        for i in range(1, len(route) - 2):  # Don't swap depot positions
            for j in range(i + 1, len(route) - 1):  # Don't swap with last depot
                if j - i == 1:  # Skip adjacent edges
                    continue
                
                # Create new route by reversing segment between i and j
                new_route = route[:i] + route[i:j+1][::-1] + route[j+1:]
                
                # Check feasibility
                if is_route_feasible(new_route, instance):
                    new_cost = calculate_route_cost(new_route, instance)
                    if new_cost < best_cost:
                        best_route = new_route
                        best_cost = new_cost
                        improved = True
                        break
            if improved:
                break
        
        if improved:
            route = best_route
    
    return best_route

def customer_exchange_improvement(solution, instance):
    """Improve customer-depot assignments using exchange moves"""
    improved = True
    best_solution = solution
    
    while improved:
        improved = False
        
        for customer in instance.customers:
            current_depot = solution.customer_assignments.get(customer)
            if not current_depot:
                continue
                
            # Try assigning customer to different depot
            for new_depot in instance.depots:
                if new_depot == current_depot:
                    continue
                
                # Create new assignment
                new_assignments = solution.customer_assignments.copy()
                new_assignments[customer] = new_depot
                
                # Rebuild routes for affected depots
                new_routes = {}
                vehicle_id = 1
                
                for depot in solution.opened_depots:
                    depot_customers = [c for c, assigned_depot in new_assignments.items() 
                                     if assigned_depot == depot]
                    
                    # Build routes for this depot
                    while depot_customers:
                        route, remaining = greedy_route_construction(depot_customers, depot, instance)
                        if route and len(route) > 2:
                            new_routes[vehicle_id] = route
                            vehicle_id += 1
                        depot_customers = remaining
                
                # Create new solution
                new_solution = LRPSolution(
                    opened_depots=solution.opened_depots,
                    customer_assignments=new_assignments,
                    routes=new_routes,
                    total_cost=0,
                    fixed_cost=0,
                    routing_cost=0
                )
                
                total_cost, fixed_cost, routing_cost = calculate_solution_cost(new_solution, instance)
                new_solution.total_cost = total_cost
                new_solution.fixed_cost = fixed_cost
                new_solution.routing_cost = routing_cost
                
                if new_solution.total_cost < best_solution.total_cost:
                    best_solution = new_solution
                    improved = True
                    break
            if improved:
                break
    
    return best_solution

def grasp_local_search(solution, instance):
    """GRASP local search phase"""
    improved_solution = solution
    
    # Step 1: Improve each route using 2-opt
    new_routes = {}
    for vehicle_id, route in solution.routes.items():
        improved_route = two_opt_route_improvement(route, instance)
        new_routes[vehicle_id] = improved_route
    
    # Update solution with improved routes
    improved_solution.routes = new_routes
    total_cost, fixed_cost, routing_cost = calculate_solution_cost(improved_solution, instance)
    improved_solution.total_cost = total_cost
    improved_solution.fixed_cost = fixed_cost
    improved_solution.routing_cost = routing_cost
    
    # Step 2: Improve customer assignments
    improved_solution = customer_exchange_improvement(improved_solution, instance)
    
    return improved_solution

print("GRASP local search phase implemented")

GRASP local search phase implemented


In [ ]:
def grasp_algorithm(instance, max_iterations=100, alpha=0.3):
    """Complete GRASP algorithm"""
    print(f"Running GRASP for {max_iterations} iterations with α={alpha}")
    
    best_solution = None
    iteration_costs = []
    iteration_times = []
    
    for iteration in range(max_iterations):
        start_time = time.time()
        
        # Construction phase
        solution = grasp_construction_phase(instance, alpha)
        
        # Local search phase
        improved_solution = grasp_local_search(solution, instance)
        
        # Update best solution
        if best_solution is None or improved_solution.total_cost < best_solution.total_cost:
            best_solution = improved_solution
        
        # Record statistics
        iteration_costs.append(improved_solution.total_cost)
        iteration_times.append(time.time() - start_time)
        
        if (iteration + 1) % 20 == 0:
            print(f"Iteration {iteration + 1}: Best cost = ${best_solution.total_cost:.2f}")
    
    return best_solution, iteration_costs, iteration_times

# Run GRASP algorithm
best_solution, iteration_costs, iteration_times = grasp_algorithm(instance, max_iterations=100, alpha=0.3)

print("\n" + "="*60)
print("GRASP RESULTS")
print("="*60)
print(f"Best Solution Cost: ${best_solution.total_cost:.2f}")
print(f"Fixed Depot Costs: ${best_solution.fixed_cost:.2f}")
print(f"Routing Costs: ${best_solution.routing_cost:.2f}")
print(f"Opened Depots: {best_solution.opened_depots}")
print(f"Customer Assignments: {best_solution.customer_assignments}")
print(f"Number of Routes: {len(best_solution.routes)}")

for vehicle_id, route in best_solution.routes.items():
    route_demand = sum(instance.demands[i] for i in route if i in instance.customers)
    utilization = (route_demand / instance.vehicle_capacity) * 100
    print(f"Route {vehicle_id}: {route} (Demand: {route_demand}, Utilization: {utilization:.1f}%)")

Running GRASP for 100 iterations with α=0.3
Iteration 20: Best cost = $141.00
Iteration 40: Best cost = $141.00
Iteration 60: Best cost = $141.00
Iteration 80: Best cost = $141.00
Iteration 100: Best cost = $141.00

GRASP RESULTS
Best Solution Cost: $141.00
Fixed Depot Costs: $100.00
Routing Costs: $41.00
Opened Depots: [4]
Customer Assignments: {1: 4, 2: 4, 3: 5}
Number of Routes: 1
Route 1: [4, 1, 2, 4] (Demand: 25, Utilization: 83.3%)


In [ ]:
def analyze_grasp_performance(iteration_costs, iteration_times):
    """Analyze GRASP performance and convergence"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    
    # Plot 1: Solution cost over iterations
    ax1.plot(iteration_costs, 'b-', alpha=0.7, linewidth=1)
    ax1.plot(range(len(iteration_costs)), 
            [min(iteration_costs[:i+1]) for i in range(len(iteration_costs))], 
            'r-', linewidth=2, label='Best so far')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Solution Cost ($)')
    ax1.set_title('GRASP Convergence')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Iteration time distribution
    ax2.hist(iteration_times, bins=20, alpha=0.7, color='green', edgecolor='black')
    ax2.set_xlabel('Iteration Time (seconds)')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Iteration Time Distribution')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Solution improvement over time
    cumulative_time = np.cumsum(iteration_times)
    best_so_far = [min(iteration_costs[:i+1]) for i in range(len(iteration_costs))]
    ax3.plot(cumulative_time, best_so_far, 'purple', linewidth=2)
    ax3.set_xlabel('Cumulative Time (seconds)')
    ax3.set_ylabel('Best Solution Cost ($)')
    ax3.set_title('Solution Quality vs Time')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Solution quality distribution
    ax4.boxplot(iteration_costs, vert=True)
    ax4.set_ylabel('Solution Cost ($)')
    ax4.set_title('Solution Quality Distribution')
    ax4.grid(True, alpha=0.3)
    
    # Add statistics
    stats_text = f"""
    Statistics:
    Mean Cost: ${np.mean(iteration_costs):.2f}
    Best Cost: ${np.min(iteration_costs):.2f}
    Worst Cost: ${np.max(iteration_costs):.2f}
    Std Dev: ${np.std(iteration_costs):.2f}
    Mean Time: {np.mean(iteration_times):.4f}s
    """
    ax4.text(1.2, 0.5, stats_text, transform=ax4.transAxes, 
             bbox=dict(boxstyle="round", facecolor='wheat', alpha=0.5),
             verticalalignment='center')
    
    plt.tight_layout()
    plt.show()
    
    # Print performance summary
    print("\nGRASP Performance Summary:")
    print(f"Total iterations: {len(iteration_costs)}")
    print(f"Best solution: ${np.min(iteration_costs):.2f}")
    print(f"Average solution: ${np.mean(iteration_costs):.2f}")
    print(f"Worst solution: ${np.max(iteration_costs):.2f}")
    print(f"Standard deviation: ${np.std(iteration_costs):.2f}")
    print(f"Average iteration time: {np.mean(iteration_times):.4f} seconds")
    print(f"Total execution time: {np.sum(iteration_times):.2f} seconds")
    print(f"Improvement from first to best: {((iteration_costs[0] - np.min(iteration_costs)) / iteration_costs[0] * 100):.1f}%")

# Analyze performance
analyze_grasp_performance(iteration_costs, iteration_times)


=== STRATEGIC OPTIMIZATION: Period 2 ===

=== DEMAND SHAPING: Period 2 ===
Demand shaping results:
  Customer 1:
    Dynamic Pricing: 26.7 units
    Behavioral Nudging: 26.7 units
    Market Segmentation: 28.9 units
    Loyalty Program: 29.7 units
  Customer 2:
    Dynamic Pricing: 29.7 units
    Behavioral Nudging: 29.7 units
    Market Segmentation: 32.1 units
  Customer 3:
    Dynamic Pricing: 18.2 units
    Behavioral Nudging: 18.2 units
    Market Segmentation: 19.7 units
    Loyalty Program: 20.2 units
  Customer 4:
    Dynamic Pricing: 24.1 units
    Behavioral Nudging: 24.1 units
    Market Segmentation: 26.0 units
  Customer 5:
    Dynamic Pricing: 30.7 units
    Behavioral Nudging: 30.7 units
    Market Segmentation: 33.2 units
    Loyalty Program: 34.1 units
  Customer 6:
    Dynamic Pricing: 30.9 units
    Market Segmentation: 33.4 units
  Customer 7:
    Dynamic Pricing: 31.2 units
    Market Segmentation: 33.8 units
    Loyalty Program: 34.7 units
  Customer 8:
    Dynam

In [ ]:
def compare_alpha_values(instance, alpha_values=[0.1, 0.3, 0.5, 0.7, 0.9], iterations=50):
    """Compare GRASP performance with different alpha values"""
    
    results = {}
    
    for alpha in alpha_values:
        print(f"Testing α={alpha}...")
        solution, costs, times = grasp_algorithm(instance, max_iterations=iterations, alpha=alpha)
        results[alpha] = {
            'best_cost': np.min(costs),
            'avg_cost': np.mean(costs),
            'std_cost': np.std(costs),
            'total_time': np.sum(times)
        }
    
    # Create comparison table
    df_comparison = pd.DataFrame(results).T
    print("\nAlpha Value Comparison:")
    print(df_comparison.round(2))
    
    # Visualize comparison
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    # Plot 1: Best cost vs alpha
    ax1.plot(alpha_values, [results[a]['best_cost'] for a in alpha_values], 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Alpha (α)')
    ax1.set_ylabel('Best Solution Cost ($)')
    ax1.set_title('Solution Quality vs RCL Parameter α')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Average cost vs alpha
    ax2.plot(alpha_values, [results[a]['avg_cost'] for a in alpha_values], 'ro-', linewidth=2, markersize=8)
    ax2.set_xlabel('Alpha (α)')
    ax2.set_ylabel('Average Solution Cost ($)')
    ax2.set_title('Average Solution Quality vs α')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Standard deviation vs alpha
    ax3.plot(alpha_values, [results[a]['std_cost'] for a in alpha_values], 'go-', linewidth=2, markersize=8)
    ax3.set_xlabel('Alpha (α)')
    ax3.set_ylabel('Standard Deviation ($)')
    ax3.set_title('Solution Diversity vs α')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Total time vs alpha
    ax4.plot(alpha_values, [results[a]['total_time'] for a in alpha_values], 'mo-', linewidth=2, markersize=8)
    ax4.set_xlabel('Alpha (α)')
    ax4.set_ylabel('Total Execution Time (seconds)')
    ax4.set_title('Computational Effort vs α')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return results

# Compare different alpha values
alpha_results = compare_alpha_values(instance, alpha_values=[0.1, 0.3, 0.5, 0.7, 0.9], iterations=50)

Testing α=0.1...
Running GRASP for 50 iterations with α=0.1
Iteration 20: Best cost = $141.00
Iteration 40: Best cost = $141.00
Testing α=0.3...
Running GRASP for 50 iterations with α=0.3
Iteration 20: Best cost = $141.00
Iteration 40: Best cost = $141.00
Testing α=0.5...
Running GRASP for 50 iterations with α=0.5
Iteration 20: Best cost = $141.00
Iteration 40: Best cost = $141.00
Testing α=0.7...
Running GRASP for 50 iterations with α=0.7
Iteration 20: Best cost = $141.00
Iteration 40: Best cost = $141.00
Testing α=0.9...
Running GRASP for 50 iterations with α=0.9
Iteration 20: Best cost = $141.00
Iteration 40: Best cost = $141.00

Alpha Value Comparison:

=== STRATEGIC OPTIMIZATION: Period 5 ===

=== DEMAND SHAPING: Period 5 ===
Demand shaping results:
  Customer 1:
    Dynamic Pricing: 25.5 units
    Behavioral Nudging: 20.9 units
    Market Segmentation: 22.6 units
    Loyalty Program: 23.3 units
  Customer 2:
    Dynamic Pricing: 32.5 units
    Behavioral Nudging: 32.7 units
    M

In [ ]:
def visualize_grasp_solution(solution, instance):
    """Visualize the best GRASP solution"""
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # Define positions
    positions = {
        1: (2, 8), 2: (4, 6), 3: (8, 7),  # Customers
        4: (3, 3), 5: (7, 3)              # Depots
    }
    
    # Plot nodes
    for node, pos in positions.items():
        if node in instance.customers:
            ax.scatter(pos[0], pos[1], s=300, c='lightblue', 
                      edgecolors='navy', linewidth=2, zorder=5)
            ax.annotate(f'C{node}\n(d={instance.demands[node]})', 
                       xy=pos, xytext=(pos[0]+0.3, pos[1]+0.3),
                       fontsize=10, fontweight='bold')
        elif node in instance.depots:
            color = 'lightgreen' if node in solution.opened_depots else 'lightgray'
            ax.scatter(pos[0], pos[1], s=400, c=color, 
                      edgecolors='darkgreen', linewidth=2, zorder=5)
            status = "OPEN" if node in solution.opened_depots else "CLOSED"
            cost = instance.depot_costs[node]
            ax.annotate(f'J{node-3}\n({status})\n(${cost})', 
                       xy=pos, xytext=(pos[0]-0.8, pos[1]-1.2),
                       fontsize=10, fontweight='bold')
    
    # Plot routes
    colors = ['red', 'blue', 'green', 'orange']
    for k, route in solution.routes.items():
        if len(route) > 1:
            color = colors[k % len(colors)]
            for i in range(len(route) - 1):
                u, v = route[i], route[i + 1]
                u_pos, v_pos = positions[u], positions[v]
                ax.plot([u_pos[0], v_pos[0]], [u_pos[1], v_pos[1]], 
                       color=color, linewidth=2, alpha=0.7, 
                       label=f'Vehicle {k}' if i == 0 else '')
                
                # Add arrow
                mid_x = (u_pos[0] + v_pos[0]) / 2
                mid_y = (u_pos[1] + v_pos[1]) / 2
                dx = v_pos[0] - u_pos[0]
                dy = v_pos[1] - u_pos[1]
                ax.annotate('', xy=(mid_x + dx*0.1, mid_y + dy*0.1), 
                           xytext=(mid_x - dx*0.1, mid_y - dy*0.1),
                           arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    
    # Plot assignments
    for customer, depot in solution.customer_assignments.items():
        if depot in solution.opened_depots:
            cust_pos = positions[customer]
            depot_pos = positions[depot]
            ax.plot([cust_pos[0], depot_pos[0]], [cust_pos[1], depot_pos[1]], 
                   'k--', alpha=0.3, linewidth=1)
    
    ax.set_xlim(-1, 10)
    ax.set_ylim(0, 10)
    ax.set_xlabel('X Coordinate', fontsize=12)
    ax.set_ylabel('Y Coordinate', fontsize=12)
    ax.set_title(f'GRASP Solution\nTotal Cost: ${solution.total_cost:.2f}', 
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

# Visualize best solution
visualize_grasp_solution(best_solution, instance)

### Why this Tier exists vs Tier 1
The GRASP heuristic addresses critical limitations of the exact mathematical approach:

**Tier 1 Limitations:**
- ❌ Exponential computational complexity
- ❌ Impractical for large instances
- ❌ Long solution times for real-world problems

**Tier 2 Advantages:**
- ✅ **Polynomial time complexity** - handles large instances efficiently
- ✅ **Randomized construction** - explores diverse solution space
- ✅ **Local search improvement** - refines solutions to near-optimality
- ✅ **Scalable approach** - works for realistic problem sizes
- ✅ **Quality-time tradeoff** - good solutions in reasonable time

### Pros / Cons vs Tier 1

**Pros:**
- ✅ **Fast execution** - seconds vs hours for large problems
- ✅ **Scalable** - handles hundreds of customers and depots
- ✅ **Solution diversity** - RCL creates different solutions each run
- ✅ **Practical applicability** - suitable for real-world use
- ✅ **Tunable parameters** - α balances exploration vs exploitation

**Cons:**
- ❌ **No optimality guarantee** - might not find the best solution
- ❌ **Parameter sensitivity** - performance depends on α and iterations
- ❌ **Solution quality variance** - different runs may give different results
- ❌ **Approximation error** - gap to optimal solution unknown

### When to use this Tier
- **Large-scale instances** where exact methods are infeasible
- **Time-critical decisions** requiring quick solutions
- **Strategic planning** with multiple scenario analyses
- **Real-time applications** needing fast responses
- **Initial solution generation** for other advanced methods
- **Benchmark development** for testing new algorithms

### Key GRASP Innovations

1. **Restricted Candidate List (RCL)**: Balances greedy choices with randomness

2. **Two-Phase Approach**: Construction + Local Search for quality improvement

3. **Iterative Framework**: Multiple iterations to find high-quality solutions

4. **Parameter Tuning**: α controls the greediness-randomness tradeoff

5. **Hybrid Improvement**: Combines route optimization (2-opt) with assignment improvement

The GRASP heuristic provides an excellent balance between solution quality and computational efficiency, making it ideal for practical Location-Routing Problem applications where exact optimization is computationally prohibitive.